# Video Game Sales Analysis — Market Trends & Platform Strategy

**Goal:** Analyze global video game sales (1980–2016) to identify top-performing platforms, genres, and regional preferences — supporting data-driven decisions for market strategy.

**Analysis period:** 2006–2016 (captures PS3/X360/Wii generation through PS4/XOne launch).

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

sns.set(style='whitegrid')

df = pd.read_csv('/datasets/games.csv')
df.columns = df.columns.str.lower()
print(f'Shape: {df.shape}')
df.head()

## 2. Data Preprocessing

In [ ]:
# Convert numeric columns stored as strings
df['user_score']      = pd.to_numeric(df['user_score'],      errors='coerce')  # 'tbd' → NaN
df['year_of_release'] = pd.to_numeric(df['year_of_release'], errors='coerce')

# Drop rows missing critical fields
df = df.dropna(subset=['year_of_release', 'rating'])

# Engineer total sales
df['total_sales'] = df[['na_sales', 'eu_sales', 'jp_sales', 'other_sales']].sum(axis=1)

print(f'Records after cleaning: {len(df):,}')
print(f'Missing values:\n{df.isna().sum()[df.isna().sum() > 0]}')

**Preprocessing decisions:**
- `user_score` was stored as object — `'tbd'` values coerced to NaN
- Rows missing `year_of_release` or `rating` dropped (uninformative for time-based and segment analysis)
- `critic_score` and `user_score` NaNs retained in the full dataset and dropped only when needed for score-based analysis

## 3. Analysis Period Selection (2006–2016)

In [ ]:
periodo = df[(df['year_of_release'] >= 2006) & (df['year_of_release'] <= 2016)].copy()
periodo['user_score'] = pd.to_numeric(periodo['user_score'], errors='coerce')

print(f'Full dataset: {len(df):,} games')
print(f'2006–2016 period: {len(periodo):,} games ({len(periodo)/len(df)*100:.1f}%)')

## 4. Platform Analysis

In [ ]:
# Platform sales totals and lifecycle
platform_sales = df.groupby('platform')['total_sales'].sum().sort_values(ascending=False)
top_platforms = platform_sales.head(8).index

# Platform evolution by year
platform_evolution = (
    df.groupby(['year_of_release', 'platform'])['total_sales']
    .sum()
    .unstack(fill_value=0)[top_platforms]
)

# Lifecycle summary
print('Platform lifecycle (first → last year with sales):')
for plat in top_platforms:
    active = platform_evolution[plat][platform_evolution[plat] > 0]
    if len(active):
        print(f'  {plat}: {active.index.min():.0f} → {active.index.max():.0f} ({active.index.max() - active.index.min():.0f} years)')

In [ ]:
# Trend 2014–2016: identify growing vs declining platforms
print('Sales trend 2014 → 2015 → 2016:')
for plat in ['PS4', 'X360', 'PS3', 'Wii']:
    vals = [platform_evolution.loc[y, plat] for y in [2014, 2015, 2016]]
    trend = '📈' if vals[2] > vals[0] else '📉'
    print(f'  {trend} {plat}: {vals[0]:.1f} → {vals[1]:.1f} → {vals[2]:.1f}')

In [ ]:
# Sales distribution per platform (2006–2016)
top10 = periodo['platform'].value_counts().head(10).index
dados_box = periodo[periodo['platform'].isin(top10)]

plt.figure(figsize=(13, 6))
sns.boxplot(data=dados_box, x='platform', y='total_sales', order=top10)
plt.title('Global Sales Distribution by Platform (2006–2016)')
plt.xlabel('Platform')
plt.ylabel('Total Sales (millions)')
plt.tight_layout()
plt.show()

**Finding:** Most platforms share similar median sales (~0.1–0.3M). The Wii is a notable outlier driven by *Wii Sports* (82.5M) — a casual gaming phenomenon. **PS4 is the only platform showing consistent sales growth through 2016.**

## 5. Genre Analysis

In [ ]:
genre_total  = periodo.groupby('genre')['total_sales'].sum().sort_values(ascending=False)
genre_avg    = periodo.groupby('genre')['total_sales'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

genre_total.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Total Sales by Genre (2006–2016)')
axes[0].set_ylabel('Total Sales (millions)')
axes[0].tick_params(axis='x', rotation=45)

genre_avg.plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Average Sales per Game by Genre (2006–2016)')
axes[1].set_ylabel('Avg Sales (millions)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

pd.DataFrame({'Total Sales (M)': genre_total.round(1), 'Avg per Game (M)': genre_avg.round(3)})

**Finding:** Action leads in total sales volume, but **Shooter games have the highest revenue per title (0.97M avg)** — making them the most efficient genre for publishers despite ranking 3rd in total revenue.

## 6. Regional Analysis

In [ ]:
regions = {'NA': 'na_sales', 'EU': 'eu_sales', 'JP': 'jp_sales'}

print('=== TOP 5 PLATFORMS BY REGION ===')
for region, col in regions.items():
    top = periodo.groupby('platform')[col].sum().sort_values(ascending=False).head(5)
    print(f'\n{region}:')
    print(top.round(1).to_string())

print('\n=== TOP 5 GENRES BY REGION ===')
for region, col in regions.items():
    top = periodo.groupby('genre')[col].sum().sort_values(ascending=False).head(5)
    print(f'\n{region}:')
    print(top.round(1).to_string())

In [ ]:
# Regional comparison chart
key_platforms = ['X360', 'PS3', 'Wii', 'DS', 'PS4']
reg_df = pd.DataFrame({
    'NA': periodo.groupby('platform')['na_sales'].sum(),
    'EU': periodo.groupby('platform')['eu_sales'].sum(),
    'JP': periodo.groupby('platform')['jp_sales'].sum()
}).loc[key_platforms]

reg_df.plot(kind='bar', figsize=(12, 6))
plt.title('Platform Sales by Region (2006–2016)')
plt.xlabel('Platform')
plt.ylabel('Sales (millions)')
plt.xticks(rotation=0)
plt.legend(title='Region')
plt.tight_layout()
plt.show()

**Finding:** Japan is a distinct market — DS/3DS dominate, Role-Playing is the top genre (vs Action elsewhere), and mature-rated games underperform vs other regions.

## 7. ESRB Rating Impact

In [ ]:
for rating in ['E', 'M']:
    subset = periodo[periodo['rating'] == rating]
    print(f"\nRating '{rating}' ({len(subset)} games):")
    for region, col in regions.items():
        avg = subset[col].sum() / len(subset)
        print(f'  {region} avg/game: {avg:.3f}M')

# M/E ratio
print('\nM vs E revenue ratio per game (M sells X times more than E):')
for region, col in regions.items():
    avg_e = periodo[periodo['rating'] == 'E'][col].mean()
    avg_m = periodo[periodo['rating'] == 'M'][col].mean()
    print(f'  {region}: {avg_m/avg_e:.2f}x')

## 8. Hypothesis Testing

**Test 1 — Xbox One vs PC: do user scores differ?**

- H₀: Mean user scores are equal for Xbox One and PC
- H₁: Mean user scores differ between platforms
- Two-tailed t-test, α = 0.05

In [ ]:
xbox_scores = periodo[periodo['platform'] == 'XOne']['user_score'].dropna()
pc_scores   = periodo[periodo['platform'] == 'PC']['user_score'].dropna()

alpha = 0.05
t1, p1 = stats.ttest_ind(xbox_scores, pc_scores)

print(f'Xbox One: N={len(xbox_scores)}, Mean={xbox_scores.mean():.2f}')
print(f'PC:       N={len(pc_scores)}, Mean={pc_scores.mean():.2f}')
print(f'T-statistic: {t1:.4f} | p-value: {p1:.6f}')
print()
if p1 < alpha:
    print(f'✅ Reject H₀ (p={p1:.4f} < {alpha}) — scores differ significantly between Xbox One and PC')
else:
    print(f'❌ Fail to reject H₀ (p={p1:.4f} ≥ {alpha})')

**Test 2 — Action vs Sports: do user scores differ?**

- H₀: Mean user scores are equal for Action and Sports genres
- H₁: Mean user scores differ between genres
- Two-tailed t-test, α = 0.05

In [ ]:
action_scores = periodo[periodo['genre'] == 'Action']['user_score'].dropna()
sports_scores = periodo[periodo['genre'] == 'Sports']['user_score'].dropna()

t2, p2 = stats.ttest_ind(action_scores, sports_scores)

print(f'Action: N={len(action_scores)}, Mean={action_scores.mean():.2f}')
print(f'Sports: N={len(sports_scores)}, Mean={sports_scores.mean():.2f}')
print(f'T-statistic: {t2:.4f} | p-value: {p2:.6f}')
print()
if p2 < alpha:
    print(f'✅ Reject H₀ (p={p2:.6f} < {alpha}) — Action and Sports user scores differ significantly')
else:
    print(f'❌ Fail to reject H₀ (p={p2:.6f} ≥ {alpha})')

## 9. Conclusion

| Analysis | Key Finding |
|---|---|
| Platform lifecycle | PS4 is the only platform growing in 2016 — strongest candidate for investment |
| Genre revenue | Action leads in volume; **Shooter has the highest avg revenue per title (0.97M)** |
| Regional markets | Japan is unique: DS/RPG dominate; mature games underperform vs NA/EU |
| ESRB ratings | Mature-rated games generate 1.59x (NA) and 1.91x (EU) more revenue per title than E-rated |
| Test 1 (platforms) | PC users score games significantly higher than Xbox One (p=0.014) |
| Test 2 (genres) | Action games rated significantly higher than Sports (p<0.0001) |

**Strategic recommendations:**
- Prioritize PS4 as the primary platform for 2017 projections
- Focus on Shooter and Action genres for maximum revenue efficiency
- Apply separate regional strategies — Japan requires a distinct approach (handheld + RPG focus)
- Consider mature-rated titles for NA and EU markets to maximize per-title revenue